In [2]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 39.3 MB/s eta 0:00:00


In [4]:
import fitz

def extract_text_from_pdf(pdf_file):
  text = ""
  with fitz.open(pdf_file) as doc:
    for page in doc:
      text += page.get_text()
  return text

pdf_text = extract_text_from_pdf("/content/sample_data/Introduction_to_Python_Programming_-_WEB.pdf")

In [5]:
from datasets import Dataset
import re

chunks = [pdf_text[i:i+1000] for i in range(0, len(pdf_text), 1000)]
data = []
for i, chunk in enumerate(chunks):
  data.append({
      "instruction": f"Summarize this textbook content (part {i+1})",
        "output": chunk
  })
print(pdf_text)
dataset = Dataset.from_list(data)
print(dataset[:100])

 
 
 
 
 
 
 
 
Introduction to Python 
Programming 
 
 
 
 
 
 
 
 
 
SENIOR CONTRIBUTING AUTHORS 
UDAYAN DAS, SAINT MARY'S COLLEGE OF CALIFORNIA 
AUBREY LAWSON, WILEY 
CHRIS MAYFIELD, JAMES MADISON UNIVERSITY 
NARGES NOROUZI, UC BERKELEY 
 
 
 
 
 
 
OpenStax 
Rice University 
6100 Main Street MS-375 
Houston, Texas 77005 
 
To learn more about OpenStax, visit https://openstax.org. 
Individual print copies and bulk orders can be purchased through our website. 
 
©2024 Rice University. Textbook content produced by OpenStax is licensed under a Creative Commons 
Attribution 4.0 International License (CC BY 4.0). Under this license, any user of this textbook or the textbook 
contents herein must provide proper attribution as follows:  
 
- 
If you redistribute this textbook in a digital format (including but not limited to PDF and HTML), then you 
must retain on every page the following attribution:  
“Access for free at openstax.org.” 
- 
If you redistribute this textbook in a print for

In [6]:
!pip install accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 8.6 MB/s eta 0:00:00


In [7]:
!pip install --upgrade transformers bitsandbytes

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
login(new_session=False)
model_name = "google/gemma-2b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
quantization_config = BitsAndBytesConfig(load_in_4bit = True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map = "auto"
)

tokenizer_config.json:   0%|          | 0.00/33.6k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [10]:
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer

lora_config = LoraConfig(
    r = 16,
    lora_alpha = 32,
    target_modules = ['q_proj', 'v_proj'],
    lora_dropout = 0.05,
    bias = 'none',
    task_type = "CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

training_args = TrainingArguments(
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_train_epochs = 1,
    learning_rate = 2e-4,
    fp16 = True,
    logging_steps = 10,
    output_dir = "./results"
)

In [11]:
from transformers import DataCollatorForLanguageModeling

def format_data(example):
  tokenized = tokenizer(
        f"### Instruction: {example['instruction']}\n### Response: {example['output']}",
        truncation=True,
        padding="max_length",
        max_length=512
    )
    # Add labels (usually same as input_ids for causal LM)
  tokenized["labels"] = tokenized["input_ids"].copy()
  return tokenized
print(len(dataset))
dataset = dataset.map(format_data)
print(dataset.column_names)

dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
trainer = Trainer(
    model = model,
    train_dataset = dataset,
    args = training_args,
    data_collator = DataCollatorForLanguageModeling(tokenizer, mlm = False),
)
trainer.train()

544


Map:   0%|          | 0/544 [00:00<?, ? examples/s]

['instruction', 'output', 'input_ids', 'attention_mask', 'labels']


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hardikjainharsora (hardikjainharsora-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,2.290100
20,2.140300
30,2.104000
40,1.843100
50,1.814000
60,1.738900
70,1.667500
80,1.647600
90,1.794200
100,1.623000


TrainOutput(global_step=136, training_loss=1.8190186865189497, metrics={'train_runtime': 493.2401, 'train_samples_per_second': 1.103, 'train_steps_per_second': 0.276, 'total_flos': 3315142112575488.0, 'train_loss': 1.8190186865189497, 'epoch': 1.0})

In [19]:
from transformers import pipeline
pipe = pipeline("text-generation",model = model, tokenizer = tokenizer)

print(pipe("What is object?", max_new_tokens = 300)[0]['generated_text'])

Device set to use cuda:0


What is object?

What is class?

What is method?

What is variable?

What is attribute?

What is method?

What is data type?

What is function?

What is value?

What is variable?

What is method?

What is function?

What is data type?

What is method?

What is data type?

What is class?

What is method?

What is class?

What is class?

What is method?

What is method?

What is class?

What is method?

What is method?

What is method?

What is class?

What is method?

What is class?

What is method?

What is method?

What is method?

What is method?

What is method?

What is class?

What is method?

What is method?

What is attribute?

What is attribute?

What is attribute?

What is attribute?

What is attribute?

What is attribute?

What is attribute?

What is attribute?

What is attribute?

What is attribute?

What is attribute?

What is attribute?

What is attribute?

What is attribute?

What is attribute?

What is attribute?

What is attribute?

What is attribute?

What is attribute

In [21]:
print(pipe(
    "### Question:\nWhat is object?\n### Answer:\n",
    max_new_tokens=150,
    do_sample=True,
    top_p=0.9,
    temperature=0.7,
    repetition_penalty=1.2,
    eos_token_id=tokenizer.eos_token_id  # stop when EOS is hit
)[0]['generated_text'])

### Question:
What is object?
### Answer:
An object in Python is an instance of a class. An object represents an entity or resource that can be referenced and manipulated by programs. Objects are created when an instance of a class is defined, such as MyClass(). The constructor is called to initialize the object.

An object has state (data) and behavior (operations). State refers to data stored within an object; for example, int variable x = 5. Behavior refers to how an object manipulates its state. For example, while(x > 0) prints "X is greater than zero."

A method is a function inside an object's definition. Methods may contain code which will run during execution of an object's lifetime. A method call consists of an object reference


In [23]:
print(pipe(
    "### Question:\nWhat are dictionaries in python and what are their applications?\n### Answer:\n",
    max_new_tokens=300,
    do_sample=True,
    top_p=0.9,
    temperature=0.7,
    repetition_penalty=1.2,
    eos_token_id=tokenizer.eos_token_id  # stop when EOS is hit
)[0]['generated_text'])

### Question:
What are dictionaries in python and what are their applications?
### Answer:
A dictionary is a data type which contains key value pairs. The keys can be strings, integers, float, Boolean, etc. Values are the corresponding values of each key. Dictionaries provide fast access to data since they store all information as key value pairs. There are many uses for dictionaries like maintaining multiple user records, storing different types of data (like number of books) based on title or ISBN.



In [14]:
from huggingface_hub import notebook_login
notebook_login()

model.push_to_hub("hardikjainharsora/textbook-model")
tokenizer.push_to_hub("hardikjainharsora/textbook-model")

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...pu7izdsgy/adapter_model.safetensors:   8%|7         |  566kB / 7.38MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmpt1_dgb5q/tokenizer.json       : 100%|##########| 34.4MB / 34.4MB            

  /tmp/tmpt1_dgb5q/tokenizer.model      : 100%|##########| 4.24MB / 4.24MB            

CommitInfo(commit_url='https://huggingface.co/hardikjainharsora/textbook-model/commit/817c6bbf01510c5a9f693a014b42725a3b53587f', commit_message='Upload tokenizer', commit_description='', oid='817c6bbf01510c5a9f693a014b42725a3b53587f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/hardikjainharsora/textbook-model', endpoint='https://huggingface.co', repo_type='model', repo_id='hardikjainharsora/textbook-model'), pr_revision=None, pr_num=None)

In [15]:
# Save locally in Colab
model.save_pretrained("./trained_model")
tokenizer.save_pretrained("./trained_model")
from google.colab import files
import shutil

# Zip the folder
shutil.make_archive("trained_model", 'zip', "./trained_model")

# Download to your PC
files.download("trained_model.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>